In [3]:
import sys
!{sys.executable} -m pip install wandb

Defaulting to user installation because normal site-packages is not writeable


In [1]:
import sys
import os

user_site = os.path.join(os.environ['APPDATA'], 'Python', 'Python313', 'site-packages')
if user_site not in sys.path:
    sys.path.append(user_site)

import wandb
print("Đã nhận thư viện wandb phiên bản:", wandb.__version__)

Đã nhận thư viện wandb phiên bản: 0.25.1


In [2]:
import wandb
wandb.login()
#wandb.login(relogin=True)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\PC\_netrc.
wandb: Currently logged in as: vhy to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [4]:
import wandb
import joblib
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

In [5]:
model_path = r'C:\Users\PC\Downloads\Diabetes-Prediction\models\model_package.joblib'
model = joblib.load(model_path)
df = pd.read_csv('../data/diabetes.csv') 

df["Glucose_BMI_Ratio"] = df["Glucose"] / (df["BMI"] + 1e-5)
df["Age_Glucose"] = df["Age"] * df["Glucose"]

df["Age_Group"] = pd.cut(df["Age"], bins=[0, 30, 45, 100], labels=[0, 1, 2]).astype(float)

df["BMI_Category"] = pd.cut(df["BMI"], bins=[0, 18.5, 25, 30, 100], labels=[0, 1, 2, 3]).astype(float)

features_list = [
    'Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 
    'DiabetesPedigreeFunction', 'Age', 'Glucose_BMI_Ratio', 'Age_Glucose', 
    'Age_Group', 'BMI_Category'
]

X = df[features_list]
y = df['Outcome']
# Lấy chính xác danh sách cột mà model 
required_columns = model.feature_names_in_.tolist()

#  Kiểm tra xem trong df hiện tại có đủ các cột
missing_cols = [c for c in required_columns if c not in df.columns]
if missing_cols:
    print(f"Đang thiếu các cột: {missing_cols}. Hãy chạy lại bước tạo feature!")
else:
    #Sắp xếp lại thứ tự cột của X y hệt như model
    X = df[required_columns]
    y = df['Outcome']
    
    # 4. Chia lại tập test
    from sklearn.model_selection import train_test_split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    print("Đã tạo X_train, y_train thành công!")
    
    print("Dữ liệu đã sẵn sàng theo đúng thứ tự model yêu cầu!")
    print("Thứ tự cột hiện tại:", X_test.columns.tolist())


D:\Users\PC\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator SimpleImputer from version 1.8.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
D:\Users\PC\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.8.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
D:\Users\PC\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.8.0 when using version 1.7.2. This might lead to breaking code or invalid results

Đã tạo X_train, y_train thành công!
Dữ liệu đã sẵn sàng theo đúng thứ tự model yêu cầu!
Thứ tự cột hiện tại: ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Glucose_BMI_Ratio', 'BMI_Category', 'Age_Group']


In [6]:

run = wandb.init(
    entity="yen-h", 
    project="RUN_Diabetes_Prediction", 
    name="upload_model",
    reinit=True
)

# model_artifact = wandb.Artifact(name="diabetes_model", type="model")
#model_path = r'C:\Users\PC\Downloads\Diabetes-Prediction\models\model_package.joblib'
#model_artifact.add_file(model_path)

# Đẩy file lên
#run.log_artifact(model_artifact)

# Kết thúc
run.finish()

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


In [7]:
X = df[features_list]
y = df['Outcome']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Danh sách các cấu hình khác nhau để so sánh
configs = [
    {"class_weight":None, "min_samples_leaf": 50, "n_estimators": 50, "max_depth": 3, "name": "small_tree1"},
    {"class_weight":"balanced", "min_samples_leaf": 20, "n_estimators": 300, "max_depth": 5, "name": "medium_tree1"},
    {"class_weight":"balanced", "min_samp`les_leaf": 1, "n_estimators": 200, "max_depth": None, "name": "deep_tree1"}
]

for config in configs:
    #  Khởi tạo W&B Run
    run = wandb.init(
        entity="yen-h",
        project="Run_Diabetes_Prediction",
        name=config["name"],
        config=config  # LOG CONFIG: Lưu Hyperparameters và Model Type
    )
    
    # Huấn luyện mô hình với Config -
    model = RandomForestClassifier(
        class_weight=config["class_weight"],
        min_samples_leaf=config["min_samples_leaf"],
        n_estimators=config["n_estimators"],
        max_depth=config["max_depth"],
        random_state=42
    )
    model.fit(X_train, y_train)

    #Dự đoán
    y_pred = model.predict(X_test)
    y_probas = model.predict_proba(X_test)
    
    # LOG METRICS: Accuracy, Precision, Recall, F1, AUC
    wandb.log({
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1_score": f1_score(y_test, y_pred),
        "auc": roc_auc_score(y_test, y_probas[:, 1])
    })
    
    # LOG CHARTS: Confusion Matrix, ROC Curve
    y_test_reset = y_test.reset_index(drop=True)
    
    wandb.log({
        "conf_mat": wandb.plot.confusion_matrix(
            probs=None,
            y_true=y_test_reset,
            preds=y_pred,
            class_names=["Healthy", "Diabetes"]
        ),
        "roc": wandb.plot.roc_curve(
            y_test_reset, 
            y_probas, 
            labels=["Healthy", "Diabetes"]
        )
    })
    
    # Kết thúc run
    run.finish()

accuracy,▁
auc,▁
f1_score,▁
precision,▁
recall,▁
accuracy,0.75974
auc,0.80937
f1_score,0.65421
precision,0.67308
recall,0.63636


accuracy,▁
auc,▁
f1_score,▁
precision,▁
recall,▁
accuracy,0.74026
auc,0.81322
f1_score,0.6875
precision,0.60274
recall,0.8


accuracy,▁
auc,▁
f1_score,▁
precision,▁
recall,▁
accuracy,0.74675
auc,0.8056
f1_score,0.64865
precision,0.64286
recall,0.65455
